## 1. Imports

In [14]:
from pathlib import Path
from typing import List, Dict, Any

import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq

import gzip
import orjson
from tqdm import tqdm

from concurrent.futures import ProcessPoolExecutor, as_completed
import os

## 2. Define directories

In [2]:
from pathlib import Path

dir_data = Path('/home/jarenas/Datasets/SemanticScholar/20260210/rawdata')
dir_parquet = Path('/home/jarenas/Datasets/SemanticScholar/20260210/parquet')

dir_parquet.mkdir(parents=True, exist_ok=True)

## 3. Import tables

### 3.1. Table **`papers`**

#### 3.1.1. Count number of papers, and infer schema

In [6]:
def print_schema_tree(schema, indent: str = ""):
    """
    Imprime un schema de Polars (df.schema) en formato árbol legible.
    Soporta Struct y List[Struct].
    """
    # Caso 1: schema es un pl.Schema (dict-like)
    if isinstance(schema, pl.Schema):
        items = list(schema.items())

    # Caso 2: schema es lista de fields (Struct.fields)
    elif isinstance(schema, list):
        items = [(f.name, f.dtype) for f in schema]

    else:
        raise TypeError(f"Unsupported schema type: {type(schema)}")

    for i, (name, dtype) in enumerate(items):
        is_last = i == len(items) - 1
        branch = "└── " if is_last else "├── "

        if isinstance(dtype, pl.Struct):
            print(f"{indent}{branch}{name}: struct")
            new_indent = indent + ("    " if is_last else "│   ")
            print_schema_tree(dtype.fields, new_indent)

        elif isinstance(dtype, pl.List) and isinstance(dtype.inner, pl.Struct):
            print(f"{indent}{branch}{name}: list<struct>")
            new_indent = indent + ("    " if is_last else "│   ")
            print_schema_tree(dtype.inner.fields, new_indent)

        else:
            print(f"{indent}{branch}{name}: {dtype}")

In [4]:
%%time

dir_papers = dir_data.joinpath('papers')
paper_files = sorted(dir_papers.glob("*.json.gz"))

print(f"Found {len(paper_files)} files")

# Contar número total de papers (sin cargarlos en memoria)
total_papers = 0
for path in tqdm(paper_files, desc="Counting papers"):
    with gzip.open(path, "rt") as f:
        for _ in f:
            total_papers += 1

print("Number of papers available:", total_papers)

# Inferimos y mostramos el esquema y algunas de las primeras filas

SAMPLE_ROWS = 5000

sample_records = []

# Solo miramos el primer fichero
first_file = paper_files[0]

with gzip.open(first_file, "rb") as f:
    for i, line in enumerate(tqdm(f, total=SAMPLE_ROWS, desc="Sampling first file for schema")):
        if i >= SAMPLE_ROWS:
            break
        try:
            sample_records.append(orjson.loads(line))
        except Exception:
            continue

print(f"Collected {len(sample_records)} sample rows from first file")

df_sample = pl.DataFrame(sample_records)

print("Schema inferido:")
print_schema_tree(df_sample.schema)

df_sample.head(2)

Found 60 files


Counting papers: 100%|█████████████████████████████████████████████████████████████████████████████████| 60/60 [11:04<00:00, 11.07s/it]


Number of papers available: 232660422


Sampling first file for schema: 100%|██████████████████████████████████████████████████████████| 5000/5000 [00:00<00:00, 141112.13it/s]

Collected 5000 sample rows from first file
Schema inferido:
├── corpusid: Int64
├── externalids: struct
│   ├── MAG: String
│   ├── CorpusId: String
│   ├── ACL: String
│   ├── PubMed: String
│   ├── DOI: String
│   ├── PubMedCentral: String
│   ├── DBLP: String
│   └── ArXiv: String
├── url: String
├── title: String
├── authors: list<struct>
│   ├── authorId: String
│   └── name: String
├── venue: String
├── publicationvenueid: String
├── year: Int64
├── referencecount: Int64
├── citationcount: Int64
├── influentialcitationcount: Int64
├── isopenaccess: Boolean
├── s2fieldsofstudy: list<struct>
│   ├── category: String
│   └── source: String
├── publicationtypes: List(String)
├── publicationdate: String
└── journal: struct
    ├── name: String
    ├── pages: String
    └── volume: String
CPU times: user 10min 50s, sys: 13.7 s, total: 11min 4s
Wall time: 11min 4s


corpusid,externalids,url,title,authors,venue,publicationvenueid,year,referencecount,citationcount,influentialcitationcount,isopenaccess,s2fieldsofstudy,publicationtypes,publicationdate,journal
i64,struct[8],str,str,list[struct[2]],str,str,i64,i64,i64,i64,bool,list[struct[2]],list[str],str,struct[3]
24107575,"{""2464673495"",""24107575"",null,""10188326"",null,null,null,null}","""https://www.semanticscholar.or…","""[Fibrinolysis in ischemic card…","[{""153019292"",""M. Cesari""}, {""36099681"",""M. Sartori""}]","""Cardiologia""","""69aeaf9a-062c-4ea8-8bd2-433fcf…",1999,0,0,0,false,"[{""Medicine"",""s2-fos-model""}, {""Medicine"",""external""}]","[""Review"", ""JournalArticle""]",null,"{""Cardiologia"","" 25-32 "",""44 1""}"
120977344,"{""1996399337"",""120977344"",null,null,""10.1119/1.1936024"",null,null,null}","""https://www.semanticscholar.or…","""Glossary Of Words And Phrases …","[{""4077862"",""L. E. Etter""}]","""""",null,2012,0,0,0,false,"[{""Medicine"",""s2-fos-model""}, {""Physics"",""s2-fos-model""}, {""Medicine"",""external""}]",null,"""2012-06-16""","{"""",null,""""}"


#### 3.1.2. Count how many articles of each type we have in the dataset

<span style="background-color:yellow;">The following code uses multiprocessing. Be careful not to use multiprocessing together with polars, as it can easily take too many CPUs (every polar process may want to use all cores), and kill the kernel/sesion.</span>

In [5]:
%%time

from collections import Counter
from concurrent.futures import ProcessPoolExecutor, as_completed
import os

def count_types_in_file(path):
    c = Counter()
    with gzip.open(path, "rb") as f:
        for line in f:
            try:
                rec = orjson.loads(line)
            except Exception:
                continue
            types = rec.get("publicationtypes")
            if types:
                c.update(types)
    return c

N_WORKERS = min(16, os.cpu_count())

counts = Counter()

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(count_types_in_file, p) for p in paper_files]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Counting publication types (parallel)"):
        counts.update(fut.result())

print(counts)

Counting publication types (parallel): 100%|███████████████████████████████████████████████████████████| 60/60 [01:13<00:00,  1.22s/it]

Counter({'JournalArticle': 59653834, 'Review': 17074096, 'Conference': 5093348, 'CaseReport': 2439286, 'Study': 2377972, 'LettersAndComments': 1629973, 'Editorial': 769585, 'ClinicalTrial': 584792, 'Book': 535455, 'News': 250327, 'MetaAnalysis': 117644, 'Dataset': 2649})
CPU times: user 79.8 ms, sys: 49.1 ms, total: 129 ms
Wall time: 1min 13s


#### 3.1.3. Selección y transformación de columnas

In [6]:
def transform_papers_df(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .select([
            pl.col("corpusid").alias("id"),

            pl.col("title"),

            pl.col("year").cast(pl.Int32),

            pl.col("externalids").struct.field("DOI").alias("doi"),
            pl.col("externalids").struct.field("PubMed").alias("pmid"),
            pl.col("externalids").struct.field("MAG").alias("magId"),
            pl.col("externalids").struct.field("ACL").alias("aclId"),

            # extractFOS: quedarnos solo con category donde source == "s2-fos-model"
            pl.when(pl.col("s2fieldsofstudy").is_not_null())
              .then(
                  pl.col("s2fieldsofstudy")
                  .list.eval(
                      pl.when(pl.element().struct.field("source") == "s2-fos-model")
                        .then(pl.element().struct.field("category"))
                  )
                  .list.drop_nulls()
              )
              .otherwise(None)
              .alias("fieldsOfStudy"),

            pl.col("publicationtypes"),

            pl.col("publicationdate"),

            pl.col("journal").struct.field("name").alias("journalName"),

            pl.col("venue"),
            pl.col("publicationvenueid"),
            pl.col("isopenaccess"),

            pl.col("referencecount").cast(pl.Int32),
            pl.col("citationcount").cast(pl.Int32),
            pl.col("influentialcitationcount").cast(pl.Int32),
        ])
    )

In [8]:
# Lo testeamos

df_transformed = transform_papers_df(df_sample)

print_schema_tree(df_transformed.schema)
df_transformed.head(2)

├── id: Int64
├── title: String
├── year: Int32
├── doi: String
├── pmid: String
├── magId: String
├── aclId: String
├── fieldsOfStudy: List(String)
├── publicationtypes: List(String)
├── publicationdate: String
├── journalName: String
├── venue: String
├── publicationvenueid: String
├── isopenaccess: Boolean
├── referencecount: Int32
├── citationcount: Int32
└── influentialcitationcount: Int32


id,title,year,doi,pmid,magId,aclId,fieldsOfStudy,publicationtypes,publicationdate,journalName,venue,publicationvenueid,isopenaccess,referencecount,citationcount,influentialcitationcount
i64,str,i32,str,str,str,str,list[str],list[str],str,str,str,str,bool,i32,i32,i32
24107575,"""[Fibrinolysis in ischemic card…",1999,null,"""10188326""","""2464673495""",null,"[""Medicine""]","[""Review"", ""JournalArticle""]",null,"""Cardiologia""","""Cardiologia""","""69aeaf9a-062c-4ea8-8bd2-433fcf…",false,0,0,0
120977344,"""Glossary Of Words And Phrases …",2012,"""10.1119/1.1936024""",null,"""1996399337""",null,"[""Medicine"", ""Physics""]",null,"""2012-06-16""","""""","""""",null,false,0,0,0


In [13]:
%%time

out_dir = dir_parquet / "papers_selected"
out_dir.mkdir(parents=True, exist_ok=True)

schema = df_sample.schema

for i, path in enumerate(tqdm(paper_files, desc="Processing papers → Parquet", unit="file")):
    df = pl.read_ndjson(
        path,
        schema=schema,               # use infered schema
        ignore_errors=True           # skip corrupt lines
    )
    df_t = transform_papers_df(df)            # column selection
    out_path = out_dir / f"papers_part_{i:05d}.parquet"
    df_t.write_parquet(out_path)              # save papers in parquet

    # libera memoria explícitamente
    del df, df_t

Processing papers → Parquet: 100%|███████████████████████████████████████████████████████████████████| 60/60 [05:55<00:00,  5.92s/file]

CPU times: user 57min, sys: 8min 15s, total: 1h 5min 15s
Wall time: 5min 55s


#### 3.1.4. Transformamos abstracts

In [15]:
def count_lines_in_gz(path: Path) -> int:
    n = 0
    with gzip.open(path, "rt") as f:
        for _ in f:
            n += 1
    return n

N_WORKERS = min(16, os.cpu_count())

dir_abstracts = dir_data.joinpath("abstracts")
abstract_files = sorted(dir_abstracts.glob("*.json.gz"))

total_abstracts = 0

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(count_lines_in_gz, p) for p in abstract_files]

    for fut in tqdm(as_completed(futures), total=len(futures), desc="Counting abstracts (parallel)"):
        total_abstracts += fut.result()

print("Number of abstracts available:", total_abstracts)

Counting abstracts (parallel): 100%|███████████████████████████████████████████████████████████████████| 30/30 [00:21<00:00,  1.42it/s]

Number of abstracts available: 37870132


In [18]:
SAMPLE_ROWS = 5000
sample_records = []

first_file = abstract_files[0]

with gzip.open(first_file, "rb") as f:
    for i, line in enumerate(tqdm(f, total=SAMPLE_ROWS, desc="Sampling abstracts schema")):
        if i >= SAMPLE_ROWS:
            break
        try:
            sample_records.append(orjson.loads(line))
        except Exception:
            continue

df_abs_sample = pl.DataFrame(sample_records)

print("Schema inferido (abstracts):")
print_schema_tree(df_abs_sample.schema)

df_abs_sample.head(2)

Sampling abstracts schema: 100%|████████████████████████████████████████████████████████████████| 5000/5000 [00:00<00:00, 78197.96it/s]

Schema inferido (abstracts):
├── corpusid: Int64
├── openaccessinfo: struct
│   ├── disclaimer: String
│   ├── externalids: struct
│   │   ├── Medline: String
│   │   ├── MAG: String
│   │   ├── ACL: String
│   │   ├── DOI: String
│   │   ├── MedRxiv: String
│   │   ├── PubMedCentral: String
│   │   └── ArXiv: String
│   ├── license: String
│   ├── url: String
│   └── status: String
└── abstract: String


corpusid,openaccessinfo,abstract
i64,struct[5],str
119304509,"{""This content is derived from https://doi.org/10.3389/fevo.2019.00133. Its open-access status is GOLD and license is CCBY."",{null,""2937112448"",null,""10.3389/fevo.2019.00133"",null,null,null},""CCBY"",""https://www.frontiersin.org/articles/10.3389/fevo.2019.00133/pdf"",""GOLD""}","""It is well known that some mem…"
190531874,"{null,{""31206836v1"",""2951767529"",null,""10.1111/aji.13156"",null,null,null},null,null,null}","""A reference range for uterine …"


In [19]:
def transform_abstracts_df(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .select([
            pl.col("corpusid").alias("id"),
            pl.col("abstract").alias("paperAbstract"),
            pl.col("openaccessinfo"),
        ])
    )

In [20]:
abs_out_dir = dir_parquet / "abstracts_selected"
abs_out_dir.mkdir(parents=True, exist_ok=True)

abs_schema = df_abs_sample.schema  # schema fijo

for i, path in enumerate(tqdm(abstract_files, desc="Processing abstracts → Parquet", unit="file")):
    df = pl.read_ndjson(
        path,
        schema=abs_schema,
        ignore_errors=True
    )
    df_t = transform_abstracts_df(df)
    out_path = abs_out_dir / f"abstracts_part_{i:05d}.parquet"
    df_t.write_parquet(out_path)

    del df, df_t

Processing abstracts → Parquet: 100%|████████████████████████████████████████████████████████████████| 30/30 [02:02<00:00,  4.07s/file]


#### 3.1.5. Hacemos el join de abstracts y papers

In [4]:
#Particionamos papers por ID

N_PARTS = 64

parts_dir_papers = dir_parquet / "papers_partitioned"
parts_dir_papers.mkdir(parents=True, exist_ok=True)

def partition_papers():
    for path in tqdm(sorted((dir_parquet / "papers_selected").glob("*.parquet")), desc="Partitioning papers"):
        df = pl.read_parquet(path)
        df = df.with_columns((pl.col("id").hash() % N_PARTS).alias("_part"))

        for part_id in range(N_PARTS):
            df_part = df.filter(pl.col("_part") == part_id).drop("_part")
            if df_part.height > 0:
                out_path = parts_dir_papers / f"part_{part_id:03d}_{path.stem}.parquet"
                df_part.write_parquet(out_path)

        del df

partition_papers()

Partitioning papers: 100%|█████████████████████████████████████████████████████████████████████████████| 60/60 [03:21<00:00,  3.36s/it]


In [5]:
#Particionamos abstracts por ID

parts_dir_abs = dir_parquet / "abstracts_partitioned"
parts_dir_abs.mkdir(parents=True, exist_ok=True)

def partition_abstracts():
    for path in tqdm(sorted((dir_parquet / "abstracts_selected").glob("*.parquet")), desc="Partitioning abstracts"):
        df = pl.read_parquet(path)
        df = df.with_columns((pl.col("id").hash() % N_PARTS).alias("_part"))

        for part_id in range(N_PARTS):
            df_part = df.filter(pl.col("_part") == part_id).drop("_part")
            if df_part.height > 0:
                out_path = parts_dir_abs / f"part_{part_id:03d}_{path.stem}.parquet"
                df_part.write_parquet(out_path)

        del df

partition_abstracts()

Partitioning abstracts: 100%|██████████████████████████████████████████████████████████████████████████| 30/30 [04:00<00:00,  8.02s/it]


In [6]:
#Join y guardamos resultado

final_dir = dir_parquet / "papers.parquet"
final_dir.mkdir(parents=True, exist_ok=True)

for part_id in tqdm(range(N_PARTS), desc="Joining partitions"):
    papers_parts = sorted(parts_dir_papers.glob(f"part_{part_id:03d}_*.parquet"))
    abs_parts = sorted(parts_dir_abs.glob(f"part_{part_id:03d}_*.parquet"))

    if not papers_parts:
        continue  # no hay papers en esta partición

    df_papers_part = pl.read_parquet(papers_parts)

    if abs_parts:
        df_abs_part = pl.read_parquet(abs_parts)
        df_joined = df_papers_part.join(df_abs_part, on="id", how="left")
        del df_abs_part
    else:
        df_joined = df_papers_part

    out_path = final_dir / f"papers_part_{part_id:03d}.parquet"
    df_joined.write_parquet(out_path)

    del df_papers_part, df_joined

Joining partitions: 100%|██████████████████████████████████████████████████████████████████████████████| 64/64 [01:23<00:00,  1.31s/it]


#### 3.1.6. Sacamos algunas estadísticas

In [3]:
# Papers totales en el dataset

total_rows = 0

for path in tqdm(sorted((dir_parquet / "papers.parquet").glob("*.parquet")), desc="Counting rows"):
    df = pl.read_parquet(path, columns=["id"])  # solo columna mínima
    total_rows += df.height
    del df

print("Total rows in final dataset:", total_rows)

Counting rows: 100%|██████████████████████████████████████████████████████████████████████████████████| 64/64 [00:00<00:00, 115.77it/s]

Total rows in final dataset: 232660422


In [4]:
# Papers con abstract
n_with_abstract = 0

for path in tqdm(sorted((dir_parquet / "papers.parquet").glob("*.parquet")), desc="Counting abstracts"):
    df = pl.read_parquet(path, columns=["paperAbstract"])
    n_with_abstract += df.filter(pl.col("paperAbstract").is_not_null()).height
    del df

print("Rows with abstract:", n_with_abstract)

Counting abstracts: 100%|██████████████████████████████████████████████████████████████████████████████| 64/64 [00:10<00:00,  6.19it/s]

Rows with abstract: 37870132


In [7]:
# Esquema
sample_path = sorted((dir_parquet / "papers.parquet").glob("*.parquet"))[0]
df_sample = pl.read_parquet(sample_path)

print_schema_tree(df_sample.schema)
df_sample.head(3)

├── id: Int64
├── title: String
├── year: Int32
├── doi: String
├── pmid: String
├── magId: String
├── aclId: String
├── fieldsOfStudy: List(String)
├── publicationtypes: List(String)
├── publicationdate: String
├── journalName: String
├── venue: String
├── publicationvenueid: String
├── isopenaccess: Boolean
├── referencecount: Int32
├── citationcount: Int32
├── influentialcitationcount: Int32
├── paperAbstract: String
└── openaccessinfo: struct
    ├── disclaimer: String
    ├── externalids: struct
    │   ├── Medline: String
    │   ├── MAG: String
    │   ├── ACL: String
    │   ├── DOI: String
    │   ├── MedRxiv: String
    │   ├── PubMedCentral: String
    │   └── ArXiv: String
    ├── license: String
    ├── url: String
    └── status: String


id,title,year,doi,pmid,magId,aclId,fieldsOfStudy,publicationtypes,publicationdate,journalName,venue,publicationvenueid,isopenaccess,referencecount,citationcount,influentialcitationcount,paperAbstract,openaccessinfo
i64,str,i32,str,str,str,str,list[str],list[str],str,str,str,str,bool,i32,i32,i32,str,struct[5]
63116831,"""Non intrusive stress and bowin…",2016,null,null,"""2565965076""",null,"[""Engineering"", ""Materials Science""]",null,"""2016-09-01""","""""","""""",null,false,0,1,0,null,null
272145529,"""METODE PENELITIAN""",null,null,null,null,null,null,null,null,null,"""""",null,false,15,0,0,null,null
157643074,"""Transfer and Transfer Studies""",2010,"""10.1075/hts.1.tra1""",null,"""2505569019""",null,null,null,"""2010-10-28""","""""","""""",null,false,3,7,1,null,null


In [8]:
# Chequeamos si hay duplicados

seen = set()
dup_count = 0

for path in tqdm(sorted((dir_parquet / "papers.parquet").glob("*.parquet")), desc="Checking duplicate ids"):
    df = pl.read_parquet(path, columns=["id"])
    for i in df["id"].to_list():
        if i in seen:
            dup_count += 1
        else:
            seen.add(i)
    del df

print("Duplicate ids found:", dup_count)

Checking duplicate ids: 100%|██████████████████████████████████████████████████████████████████████████| 64/64 [00:51<00:00,  1.23it/s]

Duplicate ids found: 0


In [9]:
import polars as pl
from tqdm import tqdm

final_paths = sorted((dir_parquet / "papers.parquet").glob("*.parquet"))

papers_with_pmid = 0
papers_with_doi = 0
unique_dois = set()   # ⚠️ ojo: puede crecer mucho; ver nota abajo

for path in tqdm(final_paths, desc="Computing PMID/DOI stats"):
    df = pl.read_parquet(path, columns=["pmid", "doi"])

    # Papers with PMID
    papers_with_pmid += df.filter(pl.col("pmid").is_not_null()).height

    # Papers with DOI
    papers_with_doi += df.filter(pl.col("doi").is_not_null()).height

    # Unique DOIs
    # (solo añadimos no nulos)
    if "doi" in df.columns:
        dois = df.filter(pl.col("doi").is_not_null()).select("doi").to_series().to_list()
        unique_dois.update(dois)

    del df

print("Papers with PMID:", papers_with_pmid)
print("Papers with DOI:", papers_with_doi)
print("Unique DOIs:", len(unique_dois))

Computing PMID/DOI stats: 100%|████████████████████████████████████████████████████████████████████████| 64/64 [00:32<00:00,  1.97it/s]

Papers with PMID: 40001016
Papers with DOI: 137422889
Unique DOIs: 136013304


In [12]:
import numpy as np

final_paths = sorted((dir_parquet / "papers.parquet").glob("*.parquet"))

word_counts_all = []

for path in tqdm(final_paths, desc="Computing abstract length distribution (words)"):
    df = pl.read_parquet(path, columns=["paperAbstract"])
    s = df.select(pl.col("paperAbstract").drop_nulls())["paperAbstract"]

    # conteo de palabras (split por espacios)
    wc = s.str.split(" ").list.len()
    word_counts_all.extend(wc.to_list())

    del df, s, wc

print("Total abstracts used:", len(word_counts_all))
print("Mean words:", np.mean(word_counts_all))
print("Median (p50):", np.percentile(word_counts_all, 50))
print("p75:", np.percentile(word_counts_all, 75))
print("p90:", np.percentile(word_counts_all, 90))
print("p99:", np.percentile(word_counts_all, 99))

Computing abstract length distribution (words): 100%|██████████████████████████████████████████████████| 64/64 [01:47<00:00,  1.68s/it]


Total abstracts used: 37870132
Mean words: 188.12918288745337
Median (p50): 180.0
p75: 237.0
p90: 297.0
p99: 524.0


#### 3.1.7. Borramos directorios y ficheros auxiliares

In [13]:
import shutil
    
base_dir = dir_parquet  # donde están tus parquets

dirs_to_delete = [
    base_dir / "papers_selected",
    base_dir / "abstracts_selected",
    base_dir / "papers_partitioned",
    base_dir / "abstracts_partitioned",
]

for d in dirs_to_delete:
    if d.exists():
        print(f"Deleting {d} ...")
        shutil.rmtree(d)
    else:
        print(f"Not found (skipped): {d}")

print("Cleanup done.")

Deleting /home/jarenas/Datasets/SemanticScholar/20260210/parquet/papers_selected ...
Deleting /home/jarenas/Datasets/SemanticScholar/20260210/parquet/abstracts_selected ...
Deleting /home/jarenas/Datasets/SemanticScholar/20260210/parquet/papers_partitioned ...
Deleting /home/jarenas/Datasets/SemanticScholar/20260210/parquet/abstracts_partitioned ...
Cleanup done.


### 3.2. Table **`authors`**

#### 3.2.1. Count number of authors and infer schema

In [15]:
%%time

dir_authors = dir_data / "authors"
author_files = sorted(dir_authors.glob("*.json.gz"))

print(f"Found {len(author_files)} author files")

def count_lines_gz(path: Path) -> int:
    n = 0
    with gzip.open(path, "rt") as f:
        for _ in f:
            n += 1
    return n

N_WORKERS = min(16, os.cpu_count())

total_authors = 0
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(count_lines_gz, p) for p in author_files]
    for fut in tqdm(as_completed(futures), total=len(futures), desc="Counting authors"):
        total_authors += fut.result()

print("Number of authors available:", total_authors)

Found 30 author files


Counting authors: 100%|████████████████████████████████████████████████████████████████████████████████| 30/30 [00:06<00:00,  4.82it/s]


Number of authors available: 113921210
CPU times: user 67.3 ms, sys: 2.04 s, total: 2.11 s
Wall time: 8.79 s


In [19]:
# Infer schema

df_auth_sample = pl.read_ndjson(
    first_file,
    infer_schema_length=50000,   # ya que sabes que todos los ficheros son grandes
    ignore_errors=True
)

print("Schema inferido (authors):")
print_schema_tree(df_auth_sample.schema)

df_auth_sample.head(2)

Schema inferido (authors):
├── authorid: String
├── externalids: struct
│   ├── DBLP: List(String)
│   └── ORCID: List(String)
├── url: String
├── name: String
├── aliases: List(String)
├── affiliations: List(String)
├── homepage: String
├── papercount: Int64
├── citationcount: Int64
└── hindex: Int64


authorid,externalids,url,name,aliases,affiliations,homepage,papercount,citationcount,hindex
str,struct[2],str,str,list[str],list[str],str,i64,i64,i64
"""2347661252""",null,"""https://www.semanticscholar.or…","""Kim G. Hankey""","[""Kim G Hankey"", ""Kim Hankey""]",null,null,7,4,1
"""2375949229""",null,"""https://www.semanticscholar.or…","""Hangtao Jin""",null,null,null,1,1,1


In [33]:
import polars as pl
from tqdm import tqdm
from pathlib import Path

author_files = sorted((dir_data / "authors").glob("*.json.gz"))

total = 0
with_orcid = 0
with_affil = 0

for path in tqdm(author_files, desc="Checking ORCID & affiliations (json.gz)"):
    df = pl.read_ndjson(
        path,
        ignore_errors=True,
        infer_schema_length=50000
    )

    total += df.height

    # ORCID presente (lista no vacía)
    with_orcid += df.filter(
        pl.col("externalids")
          .struct.field("ORCID")
          .list.len()
          .fill_null(0) > 0
    ).height

    # afiliaciones presentes (lista no vacía)
    with_affil += df.filter(
        pl.col("affiliations").list.len().fill_null(0) > 0).height

    del df

print("Total authors:", total)
print("With ORCID:", with_orcid, f"({with_orcid/total:.2%})")
print("With affiliations:", with_affil, f"({with_affil/total:.2%})")

Checking ORCID & affiliations (json.gz): 100%|█████████████████████████████████████████████████████████| 30/30 [00:22<00:00,  1.36it/s]

Total authors: 113921210
With ORCID: 44460 (0.04%)
With affiliations: 237531 (0.21%)


<span style="background-color:yellow;">Right now, it seems affiliations are missing, and ORCID also missing for most of the authors (<0.35%)? We will continue checking to see if this becomes any better.</span>

#### 3.2.2. Selección y transformación

In [34]:
%%time

dir_authors = dir_data / "authors"
author_files = sorted(dir_authors.glob("*.json.gz"))

out_dir = dir_parquet / "authors.parquet"
out_dir.mkdir(parents=True, exist_ok=True)

# 1) Inferimos schema una vez (muestreo grande para evitar Nulls)
authors_schema = pl.read_ndjson(
    author_files[0],
    infer_schema_length=50000,
    ignore_errors=True
).schema

# 2) Definimos transformación
def transform_authors_df(df: pl.DataFrame) -> pl.DataFrame:
    return df.select([
        pl.col("authorid").alias("id"),
        pl.col("name"),
        pl.col("aliases"),
        pl.col("papercount").cast(pl.Int32),
        pl.col("citationcount").cast(pl.Int32),
        pl.col("hindex").cast(pl.Int32),
    ])

# 3) Procesamos fichero a fichero → Parquet multifichero
for i, path in enumerate(tqdm(author_files, desc="Processing authors → authors.parquet", unit="file")):
    df = pl.read_ndjson(
        path,
        schema=authors_schema,
        ignore_errors=True
    )
    df_t = transform_authors_df(df)

    out_path = out_dir / f"authors_part_{i:05d}.parquet"
    df_t.write_parquet(out_path)

    del df, df_t

Processing authors → authors.parquet: 100%|██████████████████████████████████████████████████████████| 30/30 [00:23<00:00,  1.30file/s]

CPU times: user 3min 15s, sys: 14.1 s, total: 3min 29s
Wall time: 23.7 s


### 3.3. Table **`paper_author`**

In [39]:
%%time
import polars as pl
from pathlib import Path
from tqdm import tqdm

papers_files = sorted((dir_data / "papers").glob("*.json.gz"))

authors_lazy = (
    pl.scan_parquet(dir_parquet / "authors.parquet" / "*.parquet")
      .select(pl.col("id").alias("author_id"))
)

out_dir = dir_parquet / "paper_author.parquet"
out_dir.mkdir(parents=True, exist_ok=True)

total_rows = 0

papers_schema = pl.read_ndjson(
    papers_files[0],
    infer_schema_length=50000,
    ignore_errors=True
).schema

for i, path in enumerate(tqdm(papers_files, desc="Building paper_author.parquet from papers json", unit="file")):
    df = pl.read_ndjson(
        path,
        schema=papers_schema,
        ignore_errors=True
    )

    df_pa = (
        df
        .select(["corpusid", "authors"])
        .explode("authors")
        .select([
            pl.col("corpusid").alias("paper_id"),
            pl.col("authors").struct.field("authorId").alias("author_id")
        ])
    )

    # 🔧 FIX: convertir df_pa a LazyFrame antes del join
    df_pa_valid = (
        df_pa
        .lazy()
        .join(authors_lazy, on="author_id", how="inner")
        .collect()
    )

    out_path = out_dir / f"paper_author_part_{i:05d}.parquet"
    df_pa_valid.write_parquet(out_path)

    total_rows += df_pa_valid.height

    del df, df_pa, df_pa_valid

print("Number of paper_author entries (sum of parts):", total_rows)

Building paper_author.parquet from papers json: 100%|████████████████████████████████████████████████| 60/60 [07:25<00:00,  7.43s/file]

Number of paper_author entries (sum of parts): 678637276
CPU times: user 1h 2min 17s, sys: 12min 55s, total: 1h 15min 13s
Wall time: 7min 29s


### 3.4. Table **`citations`**

In [41]:
%%time
import polars as pl
from pathlib import Path
from tqdm import tqdm

dir_citations = dir_data / "citations"
citation_files = sorted(dir_citations.glob("*.json.gz"))

out_dir = dir_parquet / "citations.parquet"
out_dir.mkdir(parents=True, exist_ok=True)

# 1) Inferir schema una vez (muestreo grande para evitar Nulls)
cit_schema = pl.read_ndjson(
    citation_files[0],
    infer_schema_length=50000,
    ignore_errors=True
).schema

total_rows = 0

for i, path in enumerate(tqdm(citation_files, desc="Processing citations → citations.parquet", unit="file")):
    df = pl.read_ndjson(
        path,
        schema=cit_schema,
        ignore_errors=True
    )

    # 2) Seleccionar y renombrar columnas (equivalente a tu Spark)
    df_t = df.select([
        pl.col("citingcorpusid").alias("source"),
        pl.col("citedcorpusid").alias("dest"),
        pl.col("isinfluential"),
    ])

    out_path = out_dir / f"citations_part_{i:05d}.parquet"
    df_t.write_parquet(out_path)

    total_rows += df_t.height

    del df, df_t

print("Number of citations (sum of parts):", total_rows)

Processing citations → citations.parquet: 100%|████████████████████████████████████████████████████| 358/358 [29:02<00:00,  4.87s/file]

Number of citations (sum of parts): 5545794841
CPU times: user 3h 10s, sys: 10min 55s, total: 3h 11min 5s
Wall time: 29min 7s
